In [5]:
import os
import json
from PIL import Image
from tqdm import tqdm
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

In [ ]:
DATASET_DIR = "caer_dataset/CAER-S"
OUTPUT_JSON = "caer_dataset/output.json"
SPLITS = ["train", "test"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading Qwen3-VL model on {DEVICE}...")

model_id = "Qwen/Qwen3-VL-4B-Instruct"
cache_dir = "./models/qwen3-vl"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype="auto",
    cache_dir=cache_dir
).to(DEVICE)
processor = AutoProcessor.from_pretrained(model_id)

PROMPT_TEXT = "USER: <image>\nDescribe the surrounding environment, background context, and the person's body language or current action in detail. ASSISTANT:"

results = []

for split in SPLITS:
    split_dir = os.path.join(DATASET_DIR, split)
    if not os.path.exists(split_dir): continue

    classes = os.listdir(split_dir)

    for cls_name in classes:
        cls_dir = os.path.join(split_dir, cls_name)
        if not os.path.isdir(cls_dir): continue

        image_files = [f for f in os.listdir(cls_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Processing {split} -> {cls_name} ({len(image_files)} images)")

        for img_name in tqdm(image_files):
            img_path = os.path.join(cls_dir, img_name)
            abs_img_path = os.path.abspath(img_path)

            try:
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "image": f"file://{abs_img_path}",
                            },
                            {
                                "type": "text",
                                "text": "Describe the surrounding environment, background context, and the person's body language or current action in detail."
                            }
                        ]
                    }
                ]

                inputs = processor.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors="pt"
                )

                inputs = inputs.to(model.device)

                generated_ids = model.generate(**inputs, max_new_tokens=50)

                generated_ids_trimmed = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                ]

                output_text = processor.batch_decode(
                    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
                )[0]

                results.append({
                    "image_path": abs_img_path,
                    "split": split,
                    "label": cls_name,
                    "context_text": output_text.strip()
                })
            except Exception as e:
                print(f"Error processing {img_path}: {e}")


with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print(f"Konteks deskripsi berhasil disimpan ke {OUTPUT_JSON}")

Loading Qwen3-VL model on cuda...


Loading weights: 100%|██████████| 713/713 [00:00<00:00, 7801.11it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 11.63 GiB of which 26.81 MiB is free. Process 8013 has 10.86 GiB memory in use. Including non-PyTorch memory, this process has 646.00 MiB memory in use. Of the allocated memory 512.04 MiB is allocated by PyTorch, and 29.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import BertTokenizer

class CAERFusionDataset(Dataset):
    def __init__(self, json_path, split="train", transform=None, tokenizer=None, max_text_length=50):
        with open(json_path, 'r', encoding='utf-8') as f:
            full_data = json.load(f)

        self.data = [item for item in full_data if item['split'] == split]

        self.transform = transform
        self.tokenizer = tokenizer
        self.max_text_length = max_text_length

        self.label_map = {
            "Angry": 0,
            "Disgust": 1,
            "Fear": 2,
            "Happy": 3,
            "Sad": 4,
            "Surprise": 5,
            "Neutral": 6
        }

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        img_path = item['image_path']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        text = item['context_text']
        if self.tokenizer:
            text_inputs = self.tokenizer(
                text,
                padding='max_length',
                truncation=True,
                max_length = self.max_text_length,
                return_tensors='pt',
            )
            text_inputs = {key: val.squeeze(0) for key, val in text_inputs.items()}
        else:
            text_inputs = text

        label_str = item['label']
        label_idx = self.label_map[label_str]
        label_tensor = torch.tensor(label_idx, dtype=torch.long)

        return image, text_inputs, label_tensor
        

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
with open("caer_dataset/output.json", 'r', encoding='utf-8') as f:
    data = json.load(f)
    
train_raw = [item for item in data if item['split'] == 'train']
test_list = [item for item in data if item['split'] == 'test']

labels = [d['label'] for d in train_raw]
train_list, val_list = train_test_split(
    train_raw,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

img_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = CAERFusionDataset(
    train_list,
    transform=img_transforms,
    tokenizer=bert_tokenizer,
    max_text_length=50
)

val_dataset = CAERFusionDataset(
    val_list,
    transform=img_transforms,
    tokenizer=bert_tokenizer,
    max_text_length=50
)

test_dataset = CAERFusionDataset(
    test_list,
    transform=img_transforms,
    tokenizer=bert_tokenizer,
    max_text_length=50
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)


for batch_imgs, batch_texts, batch_labels in train_loader:
    print("Batch images shape:", batch_imgs.shape)
    print("Batch token texts:", batch_texts['input_ids'].shape)
    print("Batch labels shape:", batch_labels.shape)
    print("Contoh label index: ", batch_labels[0].item())
    break

In [ ]:
import torch.nn as nn
from transformers import ViTModel, BertModel

class XAIEmotionFusion(nn.Module):
    def __init__(self, num_classes=7, hidden_size=768, num_heads=8):
        super(XAIEmotionFusion, self).__init__()

        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_size, num_heads=num_heads, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, images, text_inputs, return_attn=False):
        vit_outputs = self.vit(pixel_values=images)
        v_features = vit_outputs.last_hidden_state

        bert_outputs = self.bert(
            input_ids=text_inputs['input_ids'],
            attention_mask=text_inputs['attention_mask']
        )

        t_features = bert_outputs.last_hidden_state

        attn_output, attn_weights = self.cross_attention(
            query=t_features,
            key=v_features,
            value=v_features
        )

        fused_features = attn_output[:, 0, :]

        out = self.dropout(fused_features)
        logits = self.classifier(out)

        if return_attn:
            return logits, attn_weights
        
        return logits

In [ ]:
model = XAIEmotionFusion().to(DEVICE)

dummy_images = torch.randn(2, 3, 224, 224).to(DEVICE)
dummy_texts = {
    'input_ids': torch.randint(0, 30522, (2, 50)).to(DEVICE),
    'attention_mask': torch.ones((2, 50), dtype=torch.long).to(DEVICE)
}

logits, attn_weights = model(dummy_images, dummy_texts, return_attn=True)

print("Logits shape:", logits.shape)
print("Attention weights shape:", attn_weights.shape)